# Cell 1 -- Installing Required Libraries

Before we begin, we need to install a few external Python libraries that are not available by default in the Kaggle environment.

- **transformers**: Provides access to pretrained deep learning models like HuBERT, which is a pretrained audio Transformer used for audio classification tasks.
- **wandb** (Weights and Biases): A tool for experiment tracking. It logs all our training metrics (loss, accuracy, F1 score, learning rate) so we can compare models easily on a dashboard.
- **librosa**: A popular Python library for audio analysis. We use it to extract handcrafted audio features like MFCCs, Chroma, and Spectral Contrast for our XGBoost baseline.
- **xgboost**: A powerful gradient-boosted tree algorithm used as our traditional ML baseline model.

The `-q` flag keeps the installation output quiet.


# Cell 1 -- Installing Required Libraries

Before we begin, we need to install a few external Python libraries that are not available by default in the Kaggle environment.

- **transformers**: Provides access to pretrained deep learning models like HuBERT, which is a pretrained audio Transformer used for audio classification tasks.
- **wandb** (Weights and Biases): A tool for experiment tracking. It logs all our training metrics (loss, accuracy, F1 score, learning rate) so we can compare models easily on a dashboard.
- **librosa**: A popular Python library for audio analysis. We use it to extract handcrafted audio features like MFCCs, Chroma, and Spectral Contrast for our XGBoost baseline.
- **xgboost**: A powerful gradient-boosted tree algorithm used as our traditional ML baseline model.

The `-q` flag keeps the installation output quiet.


In [1]:
# ============================================================
# Cell 1: Installs (Run on CPU - no GPU needed)
# ============================================================
!pip install transformers wandb librosa xgboost -q

# Cell 2 -- Imports, Seed, and Configuration

This cell does three important things:

**1. Imports:** We import all the libraries we will use throughout this notebook, including PyTorch (for building neural networks), torchaudio (for audio loading and transformations), scikit-learn (for evaluation metrics), and others.

**2. Reproducibility (Seed):** We set `SEED = 42` and apply it to Python's `random`, NumPy, and PyTorch. This ensures that every time we run the notebook, the random operations (like shuffling data or initializing weights) produce the same results, making our experiments reproducible.

**3. Constants:**
- `SAMPLE_RATE = 16000`: All audio is resampled to 16kHz. This is the standard rate for speech/audio models like HuBERT.
- `DURATION = 10`: We process 10-second audio clips. Test mashups are 30 seconds long, but our models are trained on 10-second segments.
- `NUM_CLASSES = 10`: We have 10 music genres to classify: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.
- `genre_to_idx` / `idx_to_genre`: Dictionaries to convert between genre names and numeric labels (required by PyTorch).


# Cell 2 -- Imports, Seed, and Configuration

This cell does three important things:

**1. Imports:** We import all the libraries we will use throughout this notebook, including PyTorch (for building neural networks), torchaudio (for audio loading and transformations), scikit-learn (for evaluation metrics), and others.

**2. Reproducibility (Seed):** We set `SEED = 42` and apply it to Python's `random`, NumPy, and PyTorch. This ensures that every time we run the notebook, the random operations (like shuffling data or initializing weights) produce the same results, making our experiments reproducible.

**3. Constants:**
- `SAMPLE_RATE = 16000`: All audio is resampled to 16kHz. This is the standard rate for speech/audio models like HuBERT.
- `DURATION = 10`: We process 10-second audio clips. Test mashups are 30 seconds long, but our models are trained on 10-second segments.
- `NUM_CLASSES = 10`: We have 10 music genres to classify: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.
- `genre_to_idx` / `idx_to_genre`: Dictionaries to convert between genre names and numeric labels (required by PyTorch).


In [2]:
# ============================================================
# Cell 2: Imports + Seed + Config
# ============================================================
import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split

import librosa
import librosa.display
import IPython.display as ipd

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

import wandb
from transformers import HubertForSequenceClassification
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Constants
SAMPLE_RATE = 16000
DURATION = 10         # seconds (increased from 5 for better genre recognition)
NUM_SAMPLES = SAMPLE_RATE * DURATION
NUM_CLASSES = 10

genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(genres)}
idx_to_genre = {i: g for i, g in enumerate(genres)}

print("All imports done.")
print(f"PyTorch: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

All imports done.
PyTorch: 2.8.0+cu126
GPU Available: True
GPU Name: Tesla T4


# Cell 3 -- Path Configuration and Directory Exploration

Here we set up all the file paths for our dataset and explore its structure.

**Key directories:**
- `STEMS_DIR`: Contains the training data. Each genre has 100 songs, and each song has 4 stems: `vocals.wav`, `drums.wav`, `bass.wav`, `other.wav`.
- `MASHUPS_DIR`: Contains 3020 test audio files (mashups). These are noisy mixes of stems from different songs of the same genre, with added environmental noise.
- `NOISE_DIR`: Points to the ESC-50 dataset (2000 environmental sound clips). We use this to inject realistic background noise into our training data to make our model robust to the noisy test conditions.

**Important:** The test mashups are created by mixing stems from *different* songs of the same genre, not from the same song. This insight is critical and it motivates our Cross-Song Stem Mixing augmentation strategy in later cells.


# Cell 3 -- Path Configuration and Directory Exploration

Here we set up all the file paths for our dataset and explore its structure.

**Key directories:**
- `STEMS_DIR`: Contains the training data. Each genre has 100 songs, and each song has 4 stems: `vocals.wav`, `drums.wav`, `bass.wav`, `other.wav`.
- `MASHUPS_DIR`: Contains 3020 test audio files (mashups). These are noisy mixes of stems from different songs of the same genre, with added environmental noise.
- `NOISE_DIR`: Points to the ESC-50 dataset (2000 environmental sound clips). We use this to inject realistic background noise into our training data to make our model robust to the noisy test conditions.

**Important:** The test mashups are created by mixing stems from *different* songs of the same genre, not from the same song. This insight is critical and it motivates our Cross-Song Stem Mixing augmentation strategy in later cells.


In [3]:
# ============================================================
# Cell 3: Path Config + Directory Exploration
# CRITICAL: This cell discovers the actual file naming format
# ============================================================
BASE_DIR = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'

# Fallback for older competition path structure
if not os.path.exists(BASE_DIR):
    BASE_DIR = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

STEMS_DIR = os.path.join(BASE_DIR, 'genres_stems')
MASHUPS_DIR = os.path.join(BASE_DIR, 'mashups')

# Auto-detect noise folder (ESC-50-master vs ESC-50)
if os.path.exists(os.path.join(BASE_DIR, 'ESC-50-master', 'audio')):
    NOISE_DIR = os.path.join(BASE_DIR, 'ESC-50-master', 'audio')
elif os.path.exists(os.path.join(BASE_DIR, 'ESC-50', 'audio')):
    NOISE_DIR = os.path.join(BASE_DIR, 'ESC-50', 'audio')
else:
    NOISE_DIR = None
    print("WARNING: Noise directory not found!")

print(f"BASE_DIR:   {BASE_DIR}")
print(f"STEMS_DIR:  {STEMS_DIR}")
print(f"MASHUPS_DIR:{MASHUPS_DIR}")
print(f"NOISE_DIR:  {NOISE_DIR}")
print()

# --- Explore stems directory ---
print("=" * 50)
print("TRAINING DATA (Stems per Genre)")
print("=" * 50)
for genre in genres:
    genre_path = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_path):
        songs = sorted(os.listdir(genre_path))
        # Check what stems exist in first song
        first_song_stems = os.listdir(os.path.join(genre_path, songs[0])) if songs else []
        print(f"  {genre:12s}: {len(songs)} songs | stems: {first_song_stems}")

# --- CRITICAL: Explore mashup files to discover naming format ---
print()
print("=" * 50)
print("TEST DATA (Mashup Files)")
print("=" * 50)
mashup_files = sorted(os.listdir(MASHUPS_DIR))
print(f"Total mashup files: {len(mashup_files)}")
print(f"First 10 files: {mashup_files[:10]}")
print(f"Last 5 files:   {mashup_files[-5:]}")

# --- Check test.csv ID format ---
print()
print("=" * 50)
print("test.csv and sample_submission.csv")
print("=" * 50)
test_csv_path = os.path.join(BASE_DIR, 'test.csv')
sample_sub_path = os.path.join(BASE_DIR, 'sample_submission.csv')

test_df = pd.read_csv(test_csv_path)
print(f"test.csv shape: {test_df.shape}")
print(f"test.csv columns: {list(test_df.columns)}")
print(f"First 5 rows:\n{test_df.head()}")
print(f"\nID dtype: {test_df['id'].dtype}")
print(f"Sample IDs: {test_df['id'].values[:10]}")

if os.path.exists(sample_sub_path):
    sample_sub = pd.read_csv(sample_sub_path)
    print(f"\nsample_submission.csv:\n{sample_sub.head()}")

# --- Noise files count ---
if NOISE_DIR:
    noise_files = [f for f in os.listdir(NOISE_DIR) if f.endswith('.wav')]
    print(f"\nNoise files (ESC-50): {len(noise_files)}")

BASE_DIR:   /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup
STEMS_DIR:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
MASHUPS_DIR:/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
NOISE_DIR:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio

TRAINING DATA (Stems per Genre)
  blues       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  classical   : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  country     : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  disco       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  hiphop      : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  jazz        : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  metal       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  pop         : 100 songs | stems: ['drums.wav

# Cell 9 -- GPU Device Setup

This cell sets up the GPU device for PyTorch. If a CUDA GPU is available (like Tesla T4 on Kaggle), we use it. Otherwise, we fall back to CPU. GPUs massively speed up neural network training due to their parallel processing capabilities.


# Cell 9 -- GPU Device Setup

This cell sets up the GPU device for PyTorch. If a CUDA GPU is available (like Tesla T4 on Kaggle), we use it. Otherwise, we fall back to CPU. GPUs massively speed up neural network training due to their parallel processing capabilities.


In [9]:
# ============================================================
# Cell 9: XGBoost Test Submission
# ============================================================

def extract_features_from_file(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """Extract features from a single audio file."""
    y, _ = librosa.load(file_path, sr=sr, duration=duration)
    num_frames = int(sr * duration)
    if len(y) < num_frames:
        y = np.pad(y, (0, num_frames - len(y)))
    else:
        y = y[:num_frames]
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val
    return extract_features_from_array(y, sr)


# Build file lookup for mashups (handles any naming convention)
mashup_file_lookup = {}
for f in os.listdir(MASHUPS_DIR):
    full_path = os.path.join(MASHUPS_DIR, f)
    mashup_file_lookup[f] = full_path
    name_no_ext = os.path.splitext(f)[0]
    mashup_file_lookup[name_no_ext] = full_path

print("Extracting features from test mashups...")
test_df = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))
X_test_ml = []
test_ids_ml = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test features"):
    file_id = str(row['id'])

    # Try multiple naming conventions (actual format: song0001.wav)
    file_path = None
    candidates = [
        f"song{int(file_id):04d}.wav",  # song0001.wav
        f"{file_id}.wav",
        file_id,
    ]
    # Also try using the 'filename' column if it exists
    if 'filename' in row and pd.notna(row['filename']):
        fname = os.path.basename(str(row['filename']))
        candidates.insert(0, fname)

    for c in candidates:
        if c in mashup_file_lookup:
            file_path = mashup_file_lookup[c]
            break

    if file_path is None:
        print(f"WARNING: File not found for id={file_id}, using zeros")
        X_test_ml.append(np.zeros(X_train_ml.shape[1]))
    else:
        feat = extract_features_from_file(file_path)
        X_test_ml.append(feat)
    test_ids_ml.append(int(file_id))

X_test_ml = np.array(X_test_ml)

# Predict
y_test_pred = clf.predict(X_test_ml)
y_test_genres = le.inverse_transform(y_test_pred)

# Save submission
sub_xgb = pd.DataFrame({'id': test_ids_ml, 'genre': y_test_genres})
sub_xgb.to_csv('submission_xgb.csv', index=False)
print(f"\nXGBoost submission saved: submission_xgb.csv")
print(sub_xgb.head(10))
print(f"\nGenre distribution in predictions:")
print(sub_xgb['genre'].value_counts())

Extracting features from test mashups...


Test features:   6%|▌         | 167/3020 [00:22<06:35,  7.21it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
Test features: 100%|██████████| 3020/3020 [07:00<00:00,  7.18it/s]



XGBoost submission saved: submission_xgb.csv
   id      genre
0   1        pop
1   2  classical
2   3        pop
3   4     hiphop
4   5    country
5   6     hiphop
6   7     hiphop
7   8     hiphop
8   9        pop
9  10     reggae

Genre distribution in predictions:
genre
reggae       1193
hiphop        694
pop           422
disco         188
blues         186
country       104
classical      91
jazz           66
metal          44
rock           32
Name: count, dtype: int64


# Cell 11 -- CrossSongHubertDataset (Custom Dataset for HuBERT)

This dataset class is identical to `CrossSongMelDataset` (Cell 10) in terms of the stem mixing logic, volume augmentation, noise injection, and normalization.

**The key difference:** HuBERT is a pretrained audio Transformer that processes **raw 1D waveforms** directly, not spectrograms. Therefore, this class skips the Mel Spectrogram transformation and returns the normalized 1D waveform tensor directly.


# Cell 11 -- CrossSongHubertDataset (Custom Dataset for HuBERT)

This dataset class is identical to `CrossSongMelDataset` (Cell 10) in terms of the stem mixing logic, volume augmentation, noise injection, and normalization.

**The key difference:** HuBERT is a pretrained audio Transformer that processes **raw 1D waveforms** directly, not spectrograms. Therefore, this class skips the Mel Spectrogram transformation and returns the normalized 1D waveform tensor directly.


In [11]:
# ============================================================
# Cell 11: CrossSongHubertDataset (for HuBERT — returns raw 1D waveform)
# 3x dataset multiplication for more cross-song diversity
# ============================================================

class CrossSongHubertDataset(Dataset):
    """
    Cross-song stem mixing dataset for HuBERT.
    Same mixing logic as CrossSongMelDataset but returns raw 1D waveform.
    HuBERT expects raw audio at 16kHz, not spectrograms.
    Dataset multiplied 3x for more unique cross-song combinations per epoch.
    """
    def __init__(self, stems_dir, noise_dir, genres, duration=DURATION, sample_rate=SAMPLE_RATE):
        self.sample_rate = sample_rate
        self.num_samples = sample_rate * duration
        self.stem_names = ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']

        self.genre_songs = {}
        self.all_samples = []

        for genre in genres:
            genre_path = os.path.join(stems_dir, genre)
            if os.path.exists(genre_path):
                song_paths = [os.path.join(genre_path, s) for s in os.listdir(genre_path)]
                self.genre_songs[genre] = song_paths
                for _ in song_paths:
                    self.all_samples.append(genre)

        # 3x multiplication — each repeated entry still generates a unique random mix
        base_len = len(self.all_samples)
        self.all_samples = self.all_samples * DATASET_MULTIPLIER

        self.noise_files = []
        if noise_dir and os.path.exists(noise_dir):
            self.noise_files = [os.path.join(noise_dir, f) for f in os.listdir(noise_dir) if f.endswith('.wav')]

        print(f"CrossSongHubertDataset: {len(self.all_samples)} samples ({base_len} x {DATASET_MULTIPLIER})")

    def __len__(self):
        return len(self.all_samples)

    def _load_stem(self, song_path, stem_name):
        stem_path = os.path.join(song_path, stem_name)
        if not os.path.exists(stem_path):
            return torch.zeros(1, self.num_samples)

        waveform, sr = torchaudio.load(stem_path)
        if sr != self.sample_rate:
            waveform = T.Resample(sr, self.sample_rate)(waveform)
        waveform = torch.mean(waveform, dim=0, keepdim=True)

        total_len = waveform.shape[1]
        if total_len > self.num_samples:
            start = random.randint(0, total_len - self.num_samples)
            waveform = waveform[:, start:start + self.num_samples]
        elif total_len < self.num_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.num_samples - total_len))

        return waveform

    def __getitem__(self, idx):
        genre = self.all_samples[idx]
        label = genre_to_idx[genre]
        songs_in_genre = self.genre_songs[genre]

        mixed = torch.zeros(1, self.num_samples)
        for stem_name in self.stem_names:
            random_song = random.choice(songs_in_genre)
            stem_audio = self._load_stem(random_song, stem_name)
            gain = random.uniform(0.7, 1.3)
            mixed += stem_audio * gain

        if self.noise_files and random.random() < 0.7:
            noise_wf, sr = torchaudio.load(random.choice(self.noise_files))
            if sr != self.sample_rate:
                noise_wf = T.Resample(sr, self.sample_rate)(noise_wf)
            noise_wf = torch.mean(noise_wf, dim=0, keepdim=True)
            if noise_wf.shape[1] < self.num_samples:
                noise_wf = torch.nn.functional.pad(noise_wf, (0, self.num_samples - noise_wf.shape[1]))
            else:
                noise_wf = noise_wf[:, :self.num_samples]

            signal_power = torch.mean(mixed ** 2) + 1e-10
            noise_power = torch.mean(noise_wf ** 2) + 1e-10
            snr_db = random.uniform(5, 20)
            scale = torch.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
            mixed = mixed + scale * noise_wf

        mixed = mixed / (torch.max(torch.abs(mixed)) + 1e-8)

        # Return 1D waveform (squeeze channel dim) for HuBERT
        return mixed.squeeze(0), label

print("CrossSongHubertDataset class defined (with 3x multiplier).")

CrossSongHubertDataset class defined (with 3x multiplier).


# Cell 12 -- DataLoader Setup (Train/Validation Splits)

Here we create PyTorch DataLoaders, which handle batching and shuffling during training.

**Key points:**
- Data is split 85% training / 15% validation.
- **Mel DataLoaders** (for CNN/CRNN): `batch_size=16`.
- **HuBERT DataLoaders**: `batch_size=8` (smaller because HuBERT processes raw waveforms of 160,000 samples through a large Transformer, requiring significantly more GPU memory).
- Training loaders use `shuffle=True`; validation loaders use `shuffle=False` for consistent evaluation.


# Cell 12 -- DataLoader Setup (Train/Validation Splits)

Here we create PyTorch DataLoaders, which handle batching and shuffling during training.

**Key points:**
- Data is split 85% training / 15% validation.
- **Mel DataLoaders** (for CNN/CRNN): `batch_size=16`.
- **HuBERT DataLoaders**: `batch_size=8` (smaller because HuBERT processes raw waveforms of 160,000 samples through a large Transformer, requiring significantly more GPU memory).
- Training loaders use `shuffle=True`; validation loaders use `shuffle=False` for consistent evaluation.


In [12]:
# ============================================================
# Cell 12: Build Data Loaders (GPU required from here)
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

mel_dataset = CrossSongMelDataset(STEMS_DIR, NOISE_DIR, genres)
mel_train_size = int(0.85 * len(mel_dataset))
mel_val_size = len(mel_dataset) - mel_train_size
mel_train_data, mel_val_data = random_split(mel_dataset, [mel_train_size, mel_val_size], generator=torch.Generator().manual_seed(SEED))

# Reduced batch sizes for 10s audio (2x longer = 2x more memory)
mel_train_loader = DataLoader(mel_train_data, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
mel_val_loader = DataLoader(mel_val_data, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

hubert_dataset = CrossSongHubertDataset(STEMS_DIR, NOISE_DIR, genres)
hub_train_size = int(0.85 * len(hubert_dataset))
hub_val_size = len(hubert_dataset) - hub_train_size
hub_train_data, hub_val_data = random_split(hubert_dataset, [hub_train_size, hub_val_size], generator=torch.Generator().manual_seed(SEED))

hub_train_loader = DataLoader(hub_train_data, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
hub_val_loader = DataLoader(hub_val_data, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nMel Dataset   - Train: {mel_train_size}, Val: {mel_val_size}")
print(f"HuBERT Dataset - Train: {hub_train_size}, Val: {hub_val_size}")
print(f"Duration: {DURATION}s | Mel batch: 16 | HuBERT batch: 8")

Device: cuda
CrossSongMelDataset: 3000 samples (1000 x 3), 2000 noise files
CrossSongHubertDataset: 3000 samples (1000 x 3)

Mel Dataset   - Train: 2550, Val: 450
HuBERT Dataset - Train: 2550, Val: 450
Duration: 10s | Mel batch: 16 | HuBERT batch: 8


## Milestone 5: HuBERT Fine-Tuning (Pre-trained Model)

**Strategy**: Fine-tune `superb/hubert-base-superb-ks` with phased training:
- **Phase 1 (Epochs 1–10)**: Feature encoder FROZEN — only classifier + transformer encoder train
- **Phase 2 (Epochs 11–25)**: UNFREEZE everything — full model adapts with lower LR

HuBERT expects **raw 16kHz waveform** (not spectrograms), so we use `CrossSongHubertDataset`.

# Cell 20 -- Loading HuBERT Pretrained Model (Milestone 5)

**HuBERT** (Hidden-Unit BERT) is an **Encoder-only** audio model (no decoder). It consists of:
1. **CNN Feature Encoder:** Takes raw 16kHz audio waveform and downsamples it into a sequence of feature vectors.
2. **Transformer Encoder Stack:** Processes these vectors to learn contextual audio representations, similar to how BERT understands text through masked prediction.

We load `superb/hubert-base-superb-ks` from HuggingFace with `num_labels=10` for our genres. The feature encoder is initially **frozen** (`freeze_feature_encoder()`) as Phase 1 of our phased training strategy.


# Cell 20 -- Loading HuBERT Pretrained Model (Milestone 5)

**HuBERT** (Hidden-Unit BERT) is an **Encoder-only** audio model (no decoder). It consists of:
1. **CNN Feature Encoder:** Takes raw 16kHz audio waveform and downsamples it into a sequence of feature vectors.
2. **Transformer Encoder Stack:** Processes these vectors to learn contextual audio representations, similar to how BERT understands text through masked prediction.

We load `superb/hubert-base-superb-ks` from HuggingFace with `num_labels=10` for our genres. The feature encoder is initially **frozen** (`freeze_feature_encoder()`) as Phase 1 of our phased training strategy.


In [20]:
# ============================================================
# Cell 20: Load HuBERT Model (Milestone 5)
# ============================================================

from transformers import HubertForSequenceClassification

hubert_model = HubertForSequenceClassification.from_pretrained(
    "superb/hubert-base-superb-ks",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

# PHASE 1: Freeze the feature encoder (CNN feature extractor)
# This preserves pretrained audio representations while we train the classifier
hubert_model.freeze_feature_encoder()

total_params = sum(p.numel() for p in hubert_model.parameters())
trainable_params = sum(p.numel() for p in hubert_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"HuBERT Model Loaded Successfully")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters:    {frozen_params:,}")
print(f"  Feature encoder: FROZEN (Phase 1)")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of HubertForSequenceClassification were not initialized from the model checkpoint at superb/hubert-base-superb-ks and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([12, 256]) in the checkpoint and torch.Size([10, 256]) in the model instantiated
- classifier.bias: found shape torch.Size([12]) in the checkpoint and torch.Size([10]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

HuBERT Model Loaded Successfully
  Total parameters:     94,571,159
  Trainable parameters: 90,370,711
  Frozen parameters:    4,200,448
  Feature encoder: FROZEN (Phase 1)


# Cell 21 -- HuBERT Phased Training Strategy (Milestone 5)

HuBERT is a large pretrained model. If all weights are unfrozen from the start, the large error gradients from the randomly initialized classification head would corrupt the pretrained representations (catastrophic forgetting).

**Phase 1 (Epochs 1-10):** Feature encoder stays **frozen**. Only Transformer layers and the classifier train. Optimizer: `AdamW` with `lr=3e-5` and `weight_decay=0.01`.

**Phase 2 (Epochs 11-35):** All parameters are **unfrozen**. The optimizer is rebuilt with a lower `lr=1e-5` for gentle fine-tuning. The scheduler is also reset.

**Why AdamW instead of Adam:** Both are adaptive optimizers with per-parameter learning rates. The key difference is how weight decay is applied. Adam couples weight decay with the gradient computation (L2 regularization on gradient). AdamW decouples it -- applying weight decay directly to weights during the update step. This decoupled approach gives better generalization when fine-tuning pretrained models.

**GradScaler (AMP):** FP16 has limited dynamic range, so very small gradients can underflow to zero. GradScaler multiplies the loss by a large factor before `backward()` to keep gradients representable, then unscales before `optimizer.step()`. It auto-adjusts: increases the scale when no inf/nan occurs, decreases when they do.


# Cell 21 -- HuBERT Phased Training Strategy (Milestone 5)

HuBERT is a large pretrained model. If all weights are unfrozen from the start, the large error gradients from the randomly initialized classification head would corrupt the pretrained representations (catastrophic forgetting).

**Phase 1 (Epochs 1-10):** Feature encoder stays **frozen**. Only Transformer layers and the classifier train. Optimizer: `AdamW` with `lr=3e-5` and `weight_decay=0.01`.

**Phase 2 (Epochs 11-35):** All parameters are **unfrozen**. The optimizer is rebuilt with a lower `lr=1e-5` for gentle fine-tuning. The scheduler is also reset.

**Why AdamW instead of Adam:** Both are adaptive optimizers with per-parameter learning rates. The key difference is how weight decay is applied. Adam couples weight decay with the gradient computation (L2 regularization on gradient). AdamW decouples it -- applying weight decay directly to weights during the update step. This decoupled approach gives better generalization when fine-tuning pretrained models.

**GradScaler (AMP):** FP16 has limited dynamic range, so very small gradients can underflow to zero. GradScaler multiplies the loss by a large factor before `backward()` to keep gradients representable, then unscales before `optimizer.step()`. It auto-adjusts: increases the scale when no inf/nan occurs, decreases when they do.


In [21]:
# ============================================================
# Cell 21: Train HuBERT — Phased Strategy + WandB (Milestone 5)
# Phase 1 (ep 1-10): Feature encoder FROZEN, lr=3e-5
# Phase 2 (ep 11-35): UNFROZEN, lr=1e-5
# ============================================================

wandb.init(project="22f1001611-t12026", name="hubert-cross-song-35ep-10s")

HUBERT_EPOCHS = 35
PHASE2_START = 11  # Unfreeze at this epoch

hubert_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, hubert_model.parameters()),
    lr=3e-5, weight_decay=0.01
)
hubert_scheduler = optim.lr_scheduler.CosineAnnealingLR(hubert_optimizer, T_max=HUBERT_EPOCHS)
hubert_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
hubert_scaler = torch.amp.GradScaler('cuda')

best_hubert_f1 = 0.0

print(f"Training HuBERT for {HUBERT_EPOCHS} epochs on {device} (DURATION={DURATION}s)...")
print(f"Phase 1 (ep 1-{PHASE2_START-1}): Feature encoder FROZEN, lr=3e-5")
print(f"Phase 2 (ep {PHASE2_START}-{HUBERT_EPOCHS}): UNFROZEN, lr=1e-5")
print()

for epoch in range(1, HUBERT_EPOCHS + 1):

    # --- Phase 2: Unfreeze feature encoder at epoch PHASE2_START ---
    if epoch == PHASE2_START:
        print("=" * 50)
        print("PHASE 2: Unfreezing feature encoder!")
        print("=" * 50)
        for param in hubert_model.parameters():
            param.requires_grad = True
        # Rebuild optimizer with ALL parameters + lower LR
        hubert_optimizer = optim.AdamW(
            hubert_model.parameters(), lr=1e-5, weight_decay=0.01
        )
        hubert_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            hubert_optimizer, T_max=HUBERT_EPOCHS - PHASE2_START + 1
        )
        trainable_now = sum(p.numel() for p in hubert_model.parameters() if p.requires_grad)
        print(f"Trainable parameters now: {trainable_now:,}")

    # --- Training ---
    hubert_model.train()
    train_loss, correct, total = 0.0, 0, 0

    for waveforms, labels in tqdm(hub_train_loader, desc=f"HuBERT Epoch {epoch}/{HUBERT_EPOCHS}"):
        waveforms, labels = waveforms.to(device), labels.to(device)
        hubert_optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = hubert_model(waveforms).logits
            loss = hubert_criterion(outputs, labels)

        hubert_scaler.scale(loss).backward()
        hubert_scaler.step(hubert_optimizer)
        hubert_scaler.update()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total
    hubert_scheduler.step()

    # --- Validation ---
    hubert_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for waveforms, labels in hub_val_loader:
            waveforms, labels = waveforms.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = hubert_model(waveforms).logits
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    phase = 1 if epoch < PHASE2_START else 2

    print(f"  [Phase {phase}] Loss: {train_loss/len(hub_train_loader):.4f} | "
          f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    wandb.log({
        "epoch": epoch,
        "phase": phase,
        "train_loss": train_loss / len(hub_train_loader),
        "train_acc": train_acc,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
        "lr": hubert_optimizer.param_groups[0]['lr']
    })

    if val_f1 > best_hubert_f1:
        best_hubert_f1 = val_f1
        torch.save(hubert_model.state_dict(), 'best_hubert.pth')
        print(f"  -> New best HuBERT saved (F1={val_f1:.4f})")

hubert_val_acc = val_acc
hubert_val_f1 = best_hubert_f1
wandb.finish()
print(f"\nHuBERT Training Complete. Best Val Macro F1: {best_hubert_f1:.4f}")

wandb: setting up run wfi57kw2
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260309_120638-wfi57kw2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run hubert-cross-song-35ep-10s
wandb: ⭐️ View project at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: 🚀 View run at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/wfi57kw2


Training HuBERT for 35 epochs on cuda (DURATION=10s)...
Phase 1 (ep 1-10): Feature encoder FROZEN, lr=3e-5
Phase 2 (ep 11-35): UNFROZEN, lr=1e-5




HuBERT Epoch 1/35:   0%|          | 0/319 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/p

  [Phase 1] Loss: 1.9091 | Train Acc: 0.3451 | Val Acc: 0.5333 | Val F1: 0.5232
  -> New best HuBERT saved (F1=0.5232)


HuBERT Epoch 2/35:   0%|          | 0/319 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcode

...

  [Phase 2] Loss: 0.6660 | Train Acc: 0.9345 | Val Acc: 0.9311 | Val F1: 0.9294


HuBERT Epoch 35/35:   0%|          | 0/319 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/p

  [Phase 2] Loss: 0.6750 | Train Acc: 0.9247 | Val Acc: 0.9244 | Val F1: 0.9216


wandb: 
wandb: Run history:
wandb:        epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
wandb:           lr ██████▇▇▇▇▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
wandb:        phase ▁▁▁▁▁▁▁▁▁▁█████████████████████████
wandb:    train_acc ▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████████
wandb:   train_loss █▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_acc ▁▂▄▅▆▆▅▆▇▆▇▇▇▇▇▇█▇▇▇▇▇███▇█████████
wandb: val_macro_f1 ▁▂▃▅▆▆▅▆▆▆▇▇▇▇▇▇█▇▇▇▇▇███▇█████████
wandb: 
wandb: Run summary:
wandb:        epoch 35
wandb:           lr 0
wandb:        phase 2
wandb:    train_acc 0.92471
wandb:   train_loss 0.67503
wandb:      val_acc 0.92444
wandb: val_macro_f1 0.9216
wandb: 
wandb: 🚀 View run hubert-cross-song-35ep-10s at: https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/wfi57kw2
wandb: ⭐️ View project at: https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./


HuBERT Training Complete. Best Val Macro F1: 0.9474


# Cell 22 -- HuBERT Evaluation + Full Model Comparison

Loads the best HuBERT checkpoint, evaluates on the validation set, and prints the final comparison across all milestones (XGBoost, CNN, CRNN, HuBERT) by Validation Macro F1 Score. Target is 0.80+ on the Kaggle leaderboard.


# Cell 22 -- HuBERT Evaluation + Full Model Comparison

Loads the best HuBERT checkpoint, evaluates on the validation set, and prints the final comparison across all milestones (XGBoost, CNN, CRNN, HuBERT) by Validation Macro F1 Score. Target is 0.80+ on the Kaggle leaderboard.


In [22]:
# ============================================================
# Cell 22: HuBERT Evaluation + Full Model Comparison
# ============================================================

# Load best HuBERT and get per-class report
hubert_model.load_state_dict(torch.load('best_hubert.pth'))
hubert_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for waveforms, labels in hub_val_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            outputs = hubert_model(waveforms).logits
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("HuBERT Per-Class Classification Report:")
print(classification_report(all_labels, all_preds, target_names=genres))

# --- Full Model Comparison ---
print("\n" + "=" * 55)
print("FINAL MODEL COMPARISON (All Milestones)")
print("=" * 55)
print(f"{'Model':<12} {'Val Macro F1':>14}")
print("-" * 28)
print(f"{'XGBoost':<12} {xgb_val_f1:>14.4f}")
print(f"{'CNN':<12} {best_cnn_f1:>14.4f}")
print(f"{'CRNN':<12} {best_crnn_f1:>14.4f}")
print(f"{'HuBERT':<12} {best_hubert_f1:>14.4f}")
print("-" * 28)
print(f"\nBest model: HuBERT with Macro F1 = {best_hubert_f1:.4f}")
print("Target: 0.80+ on Kaggle leaderboard")

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#t

HuBERT Per-Class Classification Report:
              precision    recall  f1-score   support

       blues       0.89      0.89      0.89        46
   classical       1.00      1.00      1.00        45
     country       0.96      0.93      0.95        46
       disco       0.95      0.91      0.93        45
      hiphop       0.98      0.92      0.95        50
        jazz       0.98      0.96      0.97        45
       metal       0.95      0.93      0.94        44
         pop       1.00      0.91      0.95        44
      reggae       0.92      0.98      0.95        49
        rock       0.76      0.94      0.84        36

    accuracy                           0.94       450
   macro avg       0.94      0.94      0.94       450
weighted avg       0.94      0.94      0.94       450


FINAL MODEL COMPARISON (All Milestones)
Model          Val Macro F1
----------------------------
XGBoost              0.6330
CNN                  0.6996
CRNN                 0.7442
HuBERT             

## Test Inference & Final Submission

**Key fix**: Robust file path handling — builds a lookup dictionary from actual files in the mashups directory and tries multiple naming conventions (with/without extension, zero-padded, etc.).

# Cell 23 -- Test Dataset Classes (Robust File Path Handling)

Defines two test-time Dataset classes:
- `TestMashupMelDataset`: Returns Mel Spectrograms for CNN/CRNN inference.
- `TestMashupHubertDataset`: Returns raw waveforms for HuBERT inference.

Both include a robust `_find_file()` method that tries multiple filename formats to handle edge cases in the Kaggle test data naming.


# Cell 23 -- Test Dataset Classes (Robust File Path Handling)

Defines two test-time Dataset classes:
- `TestMashupMelDataset`: Returns Mel Spectrograms for CNN/CRNN inference.
- `TestMashupHubertDataset`: Returns raw waveforms for HuBERT inference.

Both include a robust `_find_file()` method that tries multiple filename formats to handle edge cases in the Kaggle test data naming.


In [23]:
# ============================================================
# Cell 23: TestMashupDataset (Robust file path handling)
# ============================================================

class TestMashupMelDataset(Dataset):
    """Test dataset returning mel spectrograms (for CNN/CRNN)."""
    def __init__(self, mashups_dir, test_csv_path, sample_rate=SAMPLE_RATE, duration=DURATION):
        self.sample_rate = sample_rate
        self.num_samples = sample_rate * duration
        self.test_df = pd.read_csv(test_csv_path)

        # Build file lookup from actual directory contents
        self.available_files = {}
        for f in os.listdir(mashups_dir):
            full_path = os.path.join(mashups_dir, f)
            self.available_files[f] = full_path
            name_no_ext = os.path.splitext(f)[0]
            self.available_files[name_no_ext] = full_path

        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=1024, hop_length=512, n_mels=64
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        print(f"TestMashupMelDataset: {len(self.test_df)} test samples, "
              f"{len(os.listdir(mashups_dir))} files in directory")

    def _find_file(self, file_id, row=None):
        file_id = str(file_id)
        candidates = []
        # Try filename column first (e.g. "mashups/song0001.wav")
        if row is not None and 'filename' in row.index and pd.notna(row.get('filename')):
            fname = os.path.basename(str(row['filename']))
            candidates.append(fname)
        # Try song{id:04d}.wav format (actual Kaggle format)
        try:
            int_id = int(file_id)
            candidates.extend([
                f"song{int_id:04d}.wav",
                f"song{int_id:04d}",
            ])
        except ValueError:
            pass
        candidates.extend([f"{file_id}.wav", file_id, f"{file_id}.mp3"])
        # Try zero-padded versions without "song" prefix
        try:
            int_id = int(file_id)
            candidates.extend([
                f"{int_id:04d}.wav", f"{int_id:04d}",
                f"{int_id:03d}.wav", f"{int_id:05d}.wav",
            ])
        except ValueError:
            pass

        for c in candidates:
            if c in self.available_files:
                return self.available_files[c]
        return None

    def __len__(self):
        return len(self.test_df)

    def __getitem__(self, idx):
        row = self.test_df.iloc[idx]
        file_id = row['id']
        file_path = self._find_file(file_id, row)

        if file_path is None:
            print(f"WARNING: File not found for id={file_id}")
            mel_spec_db = torch.zeros(1, 64, 157)  # approximate shape
            return mel_spec_db, str(file_id)

        waveform, sr = torchaudio.load(file_path)
        if sr != self.sample_rate:
            waveform = T.Resample(sr, self.sample_rate)(waveform)
        waveform = torch.mean(waveform, dim=0, keepdim=True)

        if waveform.shape[1] > self.num_samples:
            waveform = waveform[:, :self.num_samples]
        elif waveform.shape[1] < self.num_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.num_samples - waveform.shape[1]))

        waveform = waveform / (torch.max(torch.abs(waveform)) + 1e-8)
        mel_spec = self.mel_transform(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)
        return mel_spec_db, str(file_id)


class TestMashupHubertDataset(Dataset):
    """Test dataset returning raw waveforms (for HuBERT)."""
    def __init__(self, mashups_dir, test_csv_path, sample_rate=SAMPLE_RATE, duration=DURATION):
        self.sample_rate = sample_rate
        self.num_samples = sample_rate * duration
        self.test_df = pd.read_csv(test_csv_path)

        self.available_files = {}
        for f in os.listdir(mashups_dir):
            full_path = os.path.join(mashups_dir, f)
            self.available_files[f] = full_path
            name_no_ext = os.path.splitext(f)[0]
            self.available_files[name_no_ext] = full_path

        print(f"TestMashupHubertDataset: {len(self.test_df)} test samples")

    def _find_file(self, file_id, row=None):
        file_id = str(file_id)
        candidates = []
        # Try filename column first (e.g. "mashups/song0001.wav")
        if row is not None and 'filename' in row.index and pd.notna(row.get('filename')):
            fname = os.path.basename(str(row['filename']))
            candidates.append(fname)
        # Try song{id:04d}.wav format (actual Kaggle format)
        try:
            int_id = int(file_id)
            candidates.extend([
                f"song{int_id:04d}.wav",
                f"song{int_id:04d}",
            ])
        except ValueError:
            pass
        candidates.extend([f"{file_id}.wav", file_id, f"{file_id}.mp3"])
        # Try zero-padded versions without "song" prefix
        try:
            int_id = int(file_id)
            candidates.extend([
                f"{int_id:04d}.wav", f"{int_id:04d}",
                f"{int_id:03d}.wav", f"{int_id:05d}.wav",
            ])
        except ValueError:
            pass

        for c in candidates:
            if c in self.available_files:
                return self.available_files[c]
        return None

    def __len__(self):
        return len(self.test_df)

    def __getitem__(self, idx):
        row = self.test_df.iloc[idx]
        file_id = row['id']
        file_path = self._find_file(file_id, row)

        if file_path is None:
            print(f"WARNING: File not found for id={file_id}")
            return torch.zeros(self.num_samples), str(file_id)

        waveform, sr = torchaudio.load(file_path)
        if sr != self.sample_rate:
            waveform = T.Resample(sr, self.sample_rate)(waveform)
        waveform = torch.mean(waveform, dim=0, keepdim=True)

        if waveform.shape[1] > self.num_samples:
            waveform = waveform[:, :self.num_samples]
        elif waveform.shape[1] < self.num_samples:
            waveform = torch.nn.functional.pad(waveform, (0, self.num_samples - waveform.shape[1]))

        waveform = waveform / (torch.max(torch.abs(waveform)) + 1e-8)
        return waveform.squeeze(0), str(file_id)


print("Test dataset classes defined (TestMashupMelDataset + TestMashupHubertDataset).")

Test dataset classes defined (TestMashupMelDataset + TestMashupHubertDataset).


# Cell 24 -- HuBERT Inference with Systematic Test Time Augmentation (TTA)

Test audio clips are 30 seconds long, but our model was trained on 10-second clips. Rather than taking a single random 10-second crop (which might miss important musical patterns), we use **Systematic TTA**.

We extract 5 overlapping 10-second windows from each 30-second clip:
- [0s-10s], [5s-15s], [10s-20s], [15s-25s], [20s-30s]

Each window passes through HuBERT independently. The **softmax probabilities** from all 5 predictions are **averaged** to produce the final prediction. This deterministically covers the entire 30 seconds, yielding more stable and reliable predictions than random cropping.

The primary submission is saved as `submission.csv`.


# Cell 24 -- HuBERT Inference with Systematic Test Time Augmentation (TTA)

Test audio clips are 30 seconds long, but our model was trained on 10-second clips. Rather than taking a single random 10-second crop (which might miss important musical patterns), we use **Systematic TTA**.

We extract 5 overlapping 10-second windows from each 30-second clip:
- [0s-10s], [5s-15s], [10s-20s], [15s-25s], [20s-30s]

Each window passes through HuBERT independently. The **softmax probabilities** from all 5 predictions are **averaged** to produce the final prediction. This deterministically covers the entire 30 seconds, yielding more stable and reliable predictions than random cropping.

The primary submission is saved as `submission.csv`.


In [24]:
# ============================================================
# Cell 24: HuBERT Test Inference + Systematic TTA + Submission
# TTA: 5 overlapping 10s windows covering full 30s audio
# Windows: [0-10, 5-15, 10-20, 15-25, 20-30] seconds
# ============================================================

test_csv_path = os.path.join(BASE_DIR, 'test.csv')

# Load best HuBERT
hubert_model.load_state_dict(torch.load('best_hubert.pth'))
hubert_model.eval()

# Systematic TTA: 5 overlapping windows covering full 30s
# At 16kHz, 10s = 160000 samples, 30s = 480000 samples
# Windows start at: 0, 5s, 10s, 15s, 20s (with 5s overlap between consecutive)
TTA_WINDOW_STARTS_SEC = [0, 5, 10, 15, 20]  # start times in seconds
TTA_CROPS = len(TTA_WINDOW_STARTS_SEC)

# Build file lookup
available_files = {}
for f in os.listdir(MASHUPS_DIR):
    full_path = os.path.join(MASHUPS_DIR, f)
    available_files[f] = full_path
    available_files[os.path.splitext(f)[0]] = full_path

def find_test_file(file_id, row=None):
    file_id = str(file_id)
    candidates = []
    if row is not None and 'filename' in row.index and pd.notna(row.get('filename')):
        candidates.append(os.path.basename(str(row['filename'])))
    try:
        int_id = int(file_id)
        candidates.extend([f"song{int_id:04d}.wav", f"song{int_id:04d}"])
    except ValueError:
        pass
    candidates.extend([f"{file_id}.wav", file_id])
    for c in candidates:
        if c in available_files:
            return available_files[c]
    return None

def load_systematic_tta_crops(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """Load full audio and extract systematic overlapping windows for TTA.
    
    Windows: [0-10s, 5-15s, 10-20s, 15-25s, 20-30s]
    This ensures EVERY part of the 30s audio is seen by the model,
    unlike random crops which may miss important sections.
    """
    waveform, file_sr = torchaudio.load(file_path)
    if file_sr != sr:
        waveform = T.Resample(file_sr, sr)(waveform)
    waveform = torch.mean(waveform, dim=0)  # mono, shape: (total_samples,)
    
    num_samples = sr * duration  # 160000 for 10s
    total_len = waveform.shape[0]
    
    crops = []
    for start_sec in TTA_WINDOW_STARTS_SEC:
        start_sample = int(start_sec * sr)
        end_sample = start_sample + num_samples
        
        if end_sample <= total_len:
            crop = waveform[start_sample:end_sample]
        elif start_sample < total_len:
            # Partial crop — pad the rest
            crop = waveform[start_sample:]
            crop = torch.nn.functional.pad(crop, (0, num_samples - crop.shape[0]))
        else:
            # Start is beyond audio length — use last possible window
            if total_len >= num_samples:
                crop = waveform[total_len - num_samples:]
            else:
                crop = torch.nn.functional.pad(waveform, (0, num_samples - total_len))
        
        crop = crop / (torch.max(torch.abs(crop)) + 1e-8)
        crops.append(crop)
    
    return torch.stack(crops)  # shape: (TTA_CROPS, num_samples)

print(f"Running HuBERT inference with SYSTEMATIC TTA ({TTA_CROPS} windows x {DURATION}s each)...")
print(f"Window starts: {TTA_WINDOW_STARTS_SEC} seconds")
test_df = pd.read_csv(test_csv_path)

all_probs = []  # Store averaged probabilities
all_ids = []

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="HuBERT Systematic TTA"):
        file_id = row['id']
        file_path = find_test_file(file_id, row)
        
        if file_path is None:
            print(f"WARNING: File not found for id={file_id}")
            all_probs.append(np.ones(NUM_CLASSES) / NUM_CLASSES)
            all_ids.append(str(file_id))
            continue
        
        # Load systematic TTA crops (covering full 30s)
        crops = load_systematic_tta_crops(file_path)  # (TTA_CROPS, num_samples)
        crops = crops.to(device)
        
        # Run all crops through model
        with torch.amp.autocast('cuda'):
            outputs = hubert_model(crops).logits  # (TTA_CROPS, NUM_CLASSES)
        
        # Average softmax probabilities across all windows
        probs = F.softmax(outputs, dim=1).mean(dim=0).cpu().numpy()
        all_probs.append(probs)
        all_ids.append(str(file_id))

# Get final predictions from averaged probabilities
all_probs = np.array(all_probs)
predictions = np.argmax(all_probs, axis=1)
pred_genres = [idx_to_genre[p] for p in predictions]

# Save submission
test_df_orig = pd.read_csv(test_csv_path)
sub_ids = [int(x) for x in all_ids] if test_df_orig['id'].dtype == int else all_ids

submission_df = pd.DataFrame({'id': sub_ids, 'genre': pred_genres})
submission_df.to_csv('submission.csv', index=False)
submission_df.to_csv('submission_hubert.csv', index=False)

# Also save the raw probabilities for ensemble use
np.save('hubert_tta_probs.npy', all_probs)

print(f"\nBest model submission saved: submission.csv (Systematic TTA x{TTA_CROPS})")
print(f"Shape: {submission_df.shape}")
print(submission_df.head(10))
print(f"\nGenre distribution:")
print(submission_df['genre'].value_counts())

Running HuBERT inference with SYSTEMATIC TTA (5 windows x 10s each)...
Window starts: [0, 5, 10, 15, 20] seconds


HuBERT Systematic TTA:   0%|          | 0/3020 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.c


Best model submission saved: submission.csv (Systematic TTA x5)
Shape: (3020, 2)
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7       rock
7   8        pop
8   9     reggae
9  10      disco

Genre distribution:
genre
reggae       358
rock         326
blues        325
hiphop       317
metal        311
disco        305
classical    298
jazz         279
pop          254
country      247
Name: count, dtype: int64


# Cell 25 -- Ensemble Submission (Weighted Soft Voting + Systematic TTA)

We combine predictions from all three deep learning models using **Weighted Soft Voting**:
- CNN: 15%, CRNN: 20%, HuBERT: 65%

HuBERT receives the largest weight because it is the strongest model. For each test sample, we multiply each model's TTA-averaged softmax probability vector by its weight, sum them, and take the argmax:

`combined = 0.15 * cnn_probs + 0.20 * crnn_probs + 0.65 * hubert_probs`

Saved as `submission_ensemble.csv`. Both this and `submission.csv` should be submitted to Kaggle.


# Cell 25 -- Ensemble Submission (Weighted Soft Voting + Systematic TTA)

We combine predictions from all three deep learning models using **Weighted Soft Voting**:
- CNN: 15%, CRNN: 20%, HuBERT: 65%

HuBERT receives the largest weight because it is the strongest model. For each test sample, we multiply each model's TTA-averaged softmax probability vector by its weight, sum them, and take the argmax:

`combined = 0.15 * cnn_probs + 0.20 * crnn_probs + 0.65 * hubert_probs`

Saved as `submission_ensemble.csv`. Both this and `submission.csv` should be submitted to Kaggle.


In [25]:
# ============================================================
# Cell 25: Ensemble Submission (Weighted Soft Voting + Systematic TTA)
# CNN + CRNN use systematic TTA too (mel spectrograms from 5 windows)
# HuBERT TTA probs already saved from Cell 24
# ============================================================

# Reload CNN and CRNN for ensemble
cnn_model = GenreCNN().to(device)
cnn_model.load_state_dict(torch.load('best_cnn.pth'))
cnn_model.eval()

crnn_model = GenreCRNN().to(device)
crnn_model.load_state_dict(torch.load('best_crnn.pth'))
crnn_model.eval()

hubert_model.load_state_dict(torch.load('best_hubert.pth'))
hubert_model.eval()

# Mel transform for CNN/CRNN inference
mel_transform = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=512, n_mels=64)
amplitude_to_db = T.AmplitudeToDB()

# Load saved HuBERT TTA probabilities (from Cell 24)
hubert_tta_probs = np.load('hubert_tta_probs.npy')  # shape: (n_test, NUM_CLASSES)

test_df = pd.read_csv(test_csv_path)
print(f"Running CNN + CRNN inference with Systematic TTA on test set ({len(test_df)} samples)...")
print(f"Window starts: {TTA_WINDOW_STARTS_SEC} seconds")

# Weights: HuBERT=0.65, CRNN=0.20, CNN=0.15
W_CNN = 0.15
W_CRNN = 0.20
W_HUBERT = 0.65

ensemble_preds = []
ensemble_ids = []
num_samples = SAMPLE_RATE * DURATION

with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Ensemble Systematic TTA"):
        file_id = row['id']
        file_path = find_test_file(file_id, row)
        
        if file_path is None:
            # Use HuBERT-only prediction
            pred = np.argmax(hubert_tta_probs[idx])
            ensemble_preds.append(idx_to_genre[pred])
            ensemble_ids.append(file_id)
            continue
        
        # Load full audio once
        waveform, sr = torchaudio.load(file_path)
        if sr != SAMPLE_RATE:
            waveform = T.Resample(sr, SAMPLE_RATE)(waveform)
        waveform = torch.mean(waveform, dim=0, keepdim=True)  # (1, total_samples)
        total_len = waveform.shape[1]
        
        # Systematic TTA for CNN/CRNN: same 5 windows as HuBERT
        cnn_probs_list = []
        crnn_probs_list = []
        
        for start_sec in TTA_WINDOW_STARTS_SEC:
            start_sample = int(start_sec * SAMPLE_RATE)
            end_sample = start_sample + num_samples
            
            if end_sample <= total_len:
                crop = waveform[:, start_sample:end_sample]
            elif start_sample < total_len:
                crop = waveform[:, start_sample:]
                crop = torch.nn.functional.pad(crop, (0, num_samples - crop.shape[1]))
            else:
                if total_len >= num_samples:
                    crop = waveform[:, total_len - num_samples:]
                else:
                    crop = torch.nn.functional.pad(waveform, (0, num_samples - total_len))
            
            crop = crop / (torch.max(torch.abs(crop)) + 1e-8)
            mel_spec = mel_transform(crop)
            mel_spec_db = amplitude_to_db(mel_spec)
            mel_input = mel_spec_db.unsqueeze(0).to(device)  # (1, 1, freq, time)
            
            with torch.amp.autocast('cuda'):
                cnn_logits = cnn_model(mel_input)
                crnn_logits = crnn_model(mel_input)
            
            cnn_probs_list.append(F.softmax(cnn_logits[0], dim=0).cpu().numpy())
            crnn_probs_list.append(F.softmax(crnn_logits[0], dim=0).cpu().numpy())
        
        # Average probs across all windows
        cnn_probs = np.mean(cnn_probs_list, axis=0)
        crnn_probs = np.mean(crnn_probs_list, axis=0)
        hub_probs = hubert_tta_probs[idx]  # Already averaged systematic TTA probs
        
        combined = W_CNN * cnn_probs + W_CRNN * crnn_probs + W_HUBERT * hub_probs
        pred = np.argmax(combined)
        ensemble_preds.append(idx_to_genre[pred])
        ensemble_ids.append(file_id)

# Save ensemble submission
sub_ensemble = pd.DataFrame({'id': ensemble_ids, 'genre': ensemble_preds})
if test_df['id'].dtype != sub_ensemble['id'].dtype:
    sub_ensemble['id'] = sub_ensemble['id'].astype(test_df['id'].dtype)

sub_ensemble.to_csv('submission_ensemble.csv', index=False)
print(f"\nEnsemble submission saved: submission_ensemble.csv")
print(f"Weights: CNN={W_CNN}, CRNN={W_CRNN}, HuBERT={W_HUBERT}")
print(f"All models use Systematic TTA ({len(TTA_WINDOW_STARTS_SEC)} windows)")
print(sub_ensemble.head(10))
print(f"\nGenre distribution (ensemble):")
print(sub_ensemble['genre'].value_counts())

# Cleanup
del cnn_model, crnn_model
gc.collect()
torch.cuda.empty_cache()

Running CNN + CRNN inference with Systematic TTA on test set (3020 samples)...
Window starts: [0, 5, 10, 15, 20] seconds


Ensemble Systematic TTA:   0%|          | 0/3020 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github


Ensemble submission saved: submission_ensemble.csv
Weights: CNN=0.15, CRNN=0.2, HuBERT=0.65
All models use Systematic TTA (5 windows)
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7       rock
7   8        pop
8   9     reggae
9  10      disco

Genre distribution (ensemble):
genre
reggae       357
blues        346
hiphop       324
rock         322
metal        321
disco        300
jazz         284
classical    271
pop          252
country      243
Name: count, dtype: int64


# Cell 26 -- WandB Final Summary

Logs a final summary to WandB comparing all milestones and prints a clean overview of what was accomplished, including all generated submission files.


# Cell 26 -- WandB Final Summary

Logs a final summary to WandB comparing all milestones and prints a clean overview of what was accomplished, including all generated submission files.


In [26]:
# ============================================================
# Cell 26: WandB Final Summary + Submission Files List
# ============================================================

wandb.init(project="22f1001611-t12026", name="final-comparison-v3")

wandb.log({
    "xgboost_val_f1": xgb_val_f1,
    "cnn_val_f1": best_cnn_f1,
    "crnn_val_f1": best_crnn_f1,
    "hubert_val_f1": best_hubert_f1,
    "best_model": "HuBERT",
    "best_val_f1": best_hubert_f1,
    "duration_seconds": DURATION,
    "tta_type": "systematic_overlapping",
    "tta_windows": len(TTA_WINDOW_STARTS_SEC),
    "dataset_multiplier": DATASET_MULTIPLIER,
})

# Create comparison table in WandB
comparison_table = wandb.Table(
    columns=["Model", "Val Macro F1", "Milestone"],
    data=[
        ["XGBoost", round(xgb_val_f1, 4), "Milestone 2"],
        ["CNN", round(best_cnn_f1, 4), "Milestone 3"],
        ["CRNN", round(best_crnn_f1, 4), "Milestone 4"],
        ["HuBERT", round(best_hubert_f1, 4), "Milestone 5"],
    ]
)
wandb.log({"model_comparison": comparison_table})

wandb.finish()

# --- Print final summary ---
print("=" * 60)
print("ALL DONE! FINAL SUMMARY (v3 — Systematic TTA + 3x Data)")
print("=" * 60)
print()
print(f"Config: DURATION={DURATION}s, TTA=Systematic {len(TTA_WINDOW_STARTS_SEC)} windows,")
print(f"        Dataset={DATASET_MULTIPLIER}x, HuBERT_EPOCHS={HUBERT_EPOCHS}")
print()
print("v3 Improvements over v2:")
print("  - Systematic TTA: 5 overlapping 10s windows [0,5,10,15,20]s")
print("    covers full 30s audio (v2 used random crops that missed sections)")
print("  - 3x training data: 3000 unique cross-song mixes per epoch")
print("    (v2 had only 1000, less diversity)")
print("  - Ensemble also uses systematic TTA for CNN/CRNN")
print()
print("Model Performance (Validation Macro F1):")
print(f"  Milestone 2 - XGBoost:  {xgb_val_f1:.4f}")
print(f"  Milestone 3 - CNN:      {best_cnn_f1:.4f}")
print(f"  Milestone 4 - CRNN:     {best_crnn_f1:.4f}")
print(f"  Milestone 5 - HuBERT:   {best_hubert_f1:.4f}")
print()
print("Submission Files Generated:")
print("  1. submission.csv          (HuBERT + Systematic TTA) <- SUBMIT THIS")
print("  2. submission_ensemble.csv (Ensemble + Systematic TTA)")
print("  3. submission_random.csv   (Milestone 1 - random baseline)")
print("  4. submission_xgb.csv      (Milestone 2 - XGBoost)")
print()
print("Submit BOTH submission.csv and submission_ensemble.csv to Kaggle.")
print("Pick whichever scores higher. Target: 0.80+ Macro F1")
print()
print("WandB Dashboard: Check your project '22f1001611-t12026' for all runs.")
print("=" * 60)

wandb: setting up run hf2axhcn
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260309_162843-hf2axhcn
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run final-comparison-v3
wandb: ⭐️ View project at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: 🚀 View run at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/hf2axhcn
wandb: updating run metadata; uploading artifact run-hf2axhcn-model_comparison
wandb: uploading artifact run-hf2axhcn-model_comparison
wandb: 
wandb: Run history:
wandb:        best_val_f1 ▁
wandb:         cnn_val_f1 ▁
wandb:        crnn_val_f1 ▁
wandb: dataset_multiplier ▁
wandb:   duration_seconds ▁
wandb:      hubert_val_f1 ▁
wandb:        tta_windows ▁
wandb:     xgboost_val_f1 ▁
wandb: 
wandb: Run summary:
wandb:         best_model HuBERT
wandb:        best_val_f1 0.9474
wandb:         cnn_val_f1 0.69959
wandb

ALL DONE! FINAL SUMMARY (v3 — Systematic TTA + 3x Data)

Config: DURATION=10s, TTA=Systematic 5 windows,
        Dataset=3x, HuBERT_EPOCHS=35

v3 Improvements over v2:
  - Systematic TTA: 5 overlapping 10s windows [0,5,10,15,20]s
    covers full 30s audio (v2 used random crops that missed sections)
  - 3x training data: 3000 unique cross-song mixes per epoch
    (v2 had only 1000, less diversity)
  - Ensemble also uses systematic TTA for CNN/CRNN

Model Performance (Validation Macro F1):
  Milestone 2 - XGBoost:  0.6330
  Milestone 3 - CNN:      0.6996
  Milestone 4 - CRNN:     0.7442
  Milestone 5 - HuBERT:   0.9474

Submission Files Generated:
  1. submission.csv          (HuBERT + Systematic TTA) <- SUBMIT THIS
  2. submission_ensemble.csv (Ensemble + Systematic TTA)
  3. submission_random.csv   (Milestone 1 - random baseline)
  4. submission_xgb.csv      (Milestone 2 - XGBoost)

Submit BOTH submission.csv and submission_ensemble.csv to Kaggle.
Pick whichever scores higher. Target: 